In [1]:
import pandas as pd
from scipy.io import mmread
import scipy.sparse as sp
import torch
import torch.nn as nn
from torch.optim import Adam
import torch.nn.functional as F
import scipy.sparse as sp
import numpy as np
import ast
import os
import matplotlib.pyplot as plt
import pyarrow.feather as feather

In [2]:
ids = pd.read_csv('data/TP4-ids.csv', index_col=0)
articles = pd.concat([
    pd.read_csv('data/TP4-articles1.csv', index_col=0),
    pd.read_csv('data/TP4-articles2.csv', index_col=0)
], ignore_index=True)

adjacency = mmread('data/TP4-matrice-adjacence.dgt').tocsr() # Shape : (67294, 67294)

### Data de validation

In [4]:
# Ensemble de validation

test_dict = {}
np.random.seed(0)
idx_val_to_del = np.random.randint(1, len(ids), size=500)
adjacency_train = adjacency.copy()
idx_val = []
for idx in idx_val_to_del :
    links = adjacency[idx-1].indices
    if len(links) > 5 :
        to_delete = np.random.choice(links, size=5, replace=False)
        adjacency_train[idx-1, to_delete] = 0
        adjacency_train.eliminate_zeros()
        test_dict[idx] = to_delete
        idx_val.append(idx)



# métriques d'évaluation

def ndcg(y_true, y_pred, k=20):
    #y_true est la liste des 20 prédictions
    #y_pred est la liste des articles effectivement cités

    dcg = 0.0
    for i, item in enumerate(y_pred):
        if item in y_true:
            rank = i+1
            dcg += 1.0 / np.log2(rank + 1)

    n_relevant = min(len(y_true), k)  #au cas où il y aurait plus de 20 articles cités
    idcg = sum([1.0 / np.log2(i+2) for i in range(n_relevant)])

    if idcg == 0.0:
        return 0.0

    ndcg = dcg / idcg
    return ndcg


def precision(y_true, y_pred):
    return len(set(y_pred) & set(y_true))/5


def evaluate_val(model, test_dict, adjacency_train, k=20):
    '''Calcule le NDCG moyen sur l'ensemble de validation'''
    model.eval()
    
    with torch.no_grad():
        E_final = model()
    
    ndcg_scores = []
    for idx, true_links in test_dict.items():
        i = idx - 1
        
        scores = (E_final @ E_final[i]).cpu().numpy()
        
        # On exclut les citations connues dans adjacency_train
        known = adjacency_train.getrow(i).indices
        scores[known] = -np.inf
        scores[i] = -np.inf
        
        top20 = np.argsort(scores)[::-1][:k]
        ndcg_score = ndcg(true_links, top20, k=k)
        ndcg_scores.append(ndcg_score)
    
    return np.mean(ndcg_scores)


c:\Users\alex\Desktop\academique\cours polymtl\log6308\tp4_bis\Paper-recommender\.venv\Lib\site-packages\scipy\sparse\_index.py:216: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


### Embeddings

In [5]:
feather_embed = pd.read_feather('data/embeddings.feather')
feather_embed.head()

feather_indexed = feather_embed.set_index('id')
embed_cols = [c for c in feather_embed.columns if c != 'id']

E0_rows = []
missing = 0
for _, row in ids.sort_values('n').iterrows():
    hex_id = row['id']
    if hex_id in feather_indexed.index:
        E0_rows.append(feather_indexed.loc[hex_id, embed_cols].values)
    else:
        E0_rows.append(np.zeros(len(embed_cols)))
        missing += 1

E0_feather = np.array(E0_rows, dtype=np.float32)
print(f"E0 shape : {E0_feather.shape}")           # (67294, 4096)
print(f"Articles manquants : {missing}")

E0 shape : (67294, 4096)
Articles manquants : 0


In [6]:
from sklearn.decomposition import PCA

print("Compression PCA des embeddings")
pca = PCA(n_components=256)
E0_compressed = pca.fit_transform(E0_feather)

Compression PCA des embeddings


# LightGCN

In [ ]:
class LightGCN_ItemItem(nn.Module):

    def __init__(self, adjacency, E0, n_layers=3):

        super().__init__()

        self.n_layers = n_layers
        n_items, embed_size = E0.shape

        # embeddings initiaux (pré-calculés)
        if E0 is not None:
            self.E0 = nn.Parameter(torch.tensor(E0, dtype=torch.float32))
        else:
            self.E0 = nn.Parameter(torch.empty(n_items, embed_size))
            nn.init.xavier_uniform_(self.E0)
        self.loss_history = []
        self.val_history = []

        # symétrisation du graphe
        A = adjacency + adjacency.T
        A = A.tocsr()
        A.data = np.ones_like(A.data)
        A.eliminate_zeros()

        self.L = self.normalize(A)


    def normalize(self, A):

        degrees = np.array(A.sum(axis=1)).flatten()

        D_inv_sqrt = sp.diags(1.0 / np.sqrt(degrees + 1e-8))
        L = D_inv_sqrt @ A @ D_inv_sqrt

        L = L.tocoo()
        indices = torch.tensor(np.vstack((L.row, L.col)),dtype=torch.long)
        values = torch.tensor(L.data, dtype=torch.float32)
        return torch.sparse_coo_tensor(indices,values,L.shape).coalesce()


    def forward(self):
        E = self.E0
        all_embeddings = [E]
        for _ in range(self.n_layers):
            E = torch.sparse.mm(self.L, E)
            all_embeddings.append(E)

        E_final = torch.stack(all_embeddings).mean(dim=0)
        return E_final
    



    def bpr_loss(self, E, i, j, k):

        e_i = E[i]
        e_j = E[j]
        e_k = E[k]

        pos = (e_i * e_j).sum(dim=1)
        neg = (e_i * e_k).sum(dim=1)

        loss = -torch.log(torch.sigmoid(pos - neg) + 1e-8).mean()

        return loss
    

    def load_checkpoint(self, checkpoint_path, device):
        if os.path.exists(checkpoint_path):
            print(f"Checkpoint trouvé, reprise depuis {checkpoint_path}")
            checkpoint = torch.load(checkpoint_path, map_location=device)
            self.load_state_dict(checkpoint['model_state'])
            self.E0 = nn.Parameter(checkpoint['E0'])
            return checkpoint
        else:
            print("Aucun checkpoint trouvé, entraînement depuis le début.")
            return None
    
    def fit(
        self,
        adjacency,
        test_dict,
        n_epochs=50,
        batch_size=2048,
        lr=1e-3,
        lambda_reg=1e-4,
        device="cuda",
        checkpoint_path=None
    ):

        device = torch.device(device if torch.cuda.is_available() else "cpu")

        self.to(device)
        self.L = self.L.to(device)

        optimizer = Adam(self.parameters(), lr=lr)
        scaler = torch.amp.GradScaler()

        start_epoch = 0
        if os.path.exists(checkpoint_path):
            print(f"Checkpoint trouvé, reprise depuis {checkpoint_path}")
            checkpoint = torch.load(checkpoint_path, map_location=device)
            self.load_state_dict(checkpoint['model_state'])
            optimizer.load_state_dict(checkpoint['optimizer_state'])
            scaler.load_state_dict(checkpoint['scaler_state'])
            start_epoch = checkpoint.get('epoch', 0) + 1
            loss_history = checkpoint.get('val_history', [])
            val_history = checkpoint.get('val_history', [])
            self.loss_history = loss_history
            self.val_history = val_history
            self.E0 = nn.Parameter(checkpoint['E0'])
            print(f"Reprise à l'epoch {start_epoch + 1}/{n_epochs}")


        rows, cols = adjacency.nonzero()

        n_pairs = len(rows)
        n_items = adjacency.shape[0]

        for epoch in range(n_epochs):
            print(f"Epoch {epoch+1}/{n_epochs}")
            self.train()

            perm = np.random.permutation(n_pairs)
            rows_shuffled = rows[perm]
            cols_shuffled = cols[perm]

            total_loss = 0
            n_batches = 0
            for start in range(0, n_pairs, batch_size):
                print(f"\rBatch {start//batch_size + 1}/{(n_pairs + batch_size - 1) // batch_size}",end="", flush=True)
                i_batch = rows_shuffled[start:start+batch_size]
                j_batch = cols_shuffled[start:start+batch_size]
                k_batch = np.random.randint(0, n_items, size=len(i_batch)) # quasiment sur que k_batch ne contient pas de paires positives (i, k) car la matrice est très creuse, mais même si c'était le cas ça ne changerait rien à l'entrainement par loi des grands nombres.

                i_t = torch.tensor(i_batch, dtype=torch.long, device=device)
                j_t = torch.tensor(j_batch, dtype=torch.long, device=device)
                k_t = torch.tensor(k_batch, dtype=torch.long, device=device)

                optimizer.zero_grad()

                # with torch.autocast(device_type="cuda", dtype=torch.float16):
                E = self.forward()
                loss_bpr = self.bpr_loss(E, i_t, j_t, k_t)
                reg = (self.E0[i_t].norm(2).pow(2) + self.E0[j_t].norm(2).pow(2) + self.E0[k_t].norm(2).pow(2)) / len(i_t)
                loss = loss_bpr + lambda_reg * reg

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                del E

                total_loss += loss.item()
                n_batches += 1

            avg_loss = total_loss / n_batches
            self.loss_history.append(avg_loss)

            val_ndcg = evaluate_val(self, test_dict, adjacency)
            self.val_history.append(val_ndcg)

            print(f"\nEpoch {epoch+1} — Loss: {avg_loss:.4f} — Val NDCG: {val_ndcg:.4f}")

            if checkpoint_path is not None:

                torch.save(
                    {
                        "epoch": epoch,
                        "model_state": self.state_dict(),
                        "optimizer_state": optimizer.state_dict(),
                        "loss_history": self.loss_history,
                        "val_history": self.val_history,
                        "E0": self.E0.detach().cpu()
                    },
                    checkpoint_path
                )

        return val_history, loss_history

In [ ]:
model = LightGCN_ItemItem(
    adjacency_train,
    E0_compressed,
    n_layers=3,
)

torch.cuda.empty_cache()
print(f"cuda available : torch.cuda.is_available()")

loss_history = model.fit(
    adjacency_train,
    test_dict,
    n_epochs=50,
    batch_size=2048,
    lr=1e-3,
    checkpoint_path="checkpoint_lightgcn_feather.pt"
)

cuda available : torch.cuda.is_available()
Epoch 1/50
Batch 303/303
Epoch 1 — Loss: 0.3285 — Val NDCG: 0.1202
Epoch 2/50
Batch 303/303
Epoch 2 — Loss: 0.0891 — Val NDCG: 0.1309
Epoch 3/50
Batch 275/303